In [ ]:
from src.post_processing import PathWrangler
import polars as pl
from pathlib import Path
from collections import defaultdict
from ergochemics.draw import draw_reaction, draw_molecule
from ergochemics.mapping import get_reaction_center
from ergochemics.similarity import MolFeaturizer, ReactionFingerprinter
from rdkit import Chem
from IPython.display import display, SVG
import numpy as np
from hydra import initialize, compose

In [31]:
# study = "/home/stef/quest_data/bottle/data/processed/260327_target_3hpa"
study = "/home/stef/quest_data/bottle/data/processed/260410_target_3hpa"
# study = "/projects/b1039/spn1560/bottle/data/processed/260327_target_3hpa"
# study = "/home/stef/quest_data/bottle/data/processed/260324_2_step_bi_akg_to_hopa"
known = "/home/stef/bottle/artifacts/known"
# out_dir = "/home/stef/bottle/artifacts/coa_mutase_paths"

Direct look at parquet files

In [ ]:
with initialize(config_path="../conf/filepaths", version_base=None):
    filepaths = compose(config_name="filepaths")

In [ ]:
cpds = pl.read_parquet(
    Path(study)  / "compounds.parquet"
)
cpds.head()

In [ ]:
paths = pl.read_parquet(
    Path(study) / "paths.parquet"
)
paths.head()

In [ ]:
path_stats = pl.read_parquet(
    Path(study) / "path_stats.parquet"
)
path_stats.head()

In [ ]:
print(paths.shape)
print(paths.unique().shape)
print(path_stats.shape)
print(path_stats.unique().shape)

In [ ]:
pred_rxns = pl.read_parquet(
    Path(study) / "predicted_reactions.parquet"
)
print(pred_rxns.shape)
pred_rxns.head()

In [32]:
path_fb = pl.read_parquet(
    Path(study) / "path_feedback.parquet"
)
path_fb.head()

username,id,feedback,date,time
str,str,i32,date,time
"""Stefan""","""679ce8c6e93c5c6e2f1d54772b57ee…",0,2026-04-17,10:51:12.290072
"""Stefan""","""0b7bb08c699db5934fb346b2b62807…",0,2026-04-17,10:51:12.290072
"""Stefan""","""dfad596a72a8de6dc5321e895bc07f…",0,2026-04-17,10:51:12.290072
"""Stefan""","""787c56aee7410d3a28b4b7b22a5536…",0,2026-04-17,10:51:12.290072
"""Stefan""","""2f65915a529f9a86773e687909be2f…",1,2026-04-17,10:51:12.290072


In [33]:
path_fb.shape

(45, 5)

In [34]:
rxn_fb = pl.read_parquet(
    Path(study) / "reaction_feedback.parquet"
)
rxn_fb.head()

username,id,feedback,date,time
str,str,i32,date,time
"""Stefan""","""5959dace01dac3a519f9d50bb3af57…",0,2026-04-16,16:15:55.825690
"""Stefan""","""fb4e2a8027e11dbec27fe6ca29a6bc…",0,2026-04-16,16:15:55.825690
"""Stefan""","""dd322af258382cbc5680502046c37b…",0,2026-04-16,16:15:55.825690
"""Stefan""","""9038f7a31b67d543ce064375f4050b…",0,2026-04-16,16:15:55.825690
"""Stefan""","""cecd6de9246a45d052a60c0829fe77…",0,2026-04-16,16:15:55.825690


In [2]:
import sqlite3

# Connect to the database
conn = sqlite3.connect('/home/stef/quest_data/bottle/data/feedback/feedback.db')
cursor = conn.cursor()

# Query to fetch all table names
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

for table in tables:
    print(table[0])


path_feedback
reaction_feedback


In [11]:
cursor.execute("SELECT * FROM reaction_feedback")
reaction_feedback_rows = cursor.fetchall()
reaction_feedback_rows

[('stefanpate94@gmail.com',
  '260324_2_step_bi_akg_to_hopa',
  'f39c8cbe03e24a5466d6f112c736e3251e110f69',
  1,
  '2026-03-27T11:04:43.503765'),
 ('stefanpate94@gmail.com',
  '260324_2_step_bi_akg_to_hopa',
  'a5d3aa5d707970ebd6fcf2fe358248d1a69cacad',
  1,
  '2026-03-27T11:04:43.503765'),
 ('stefanpate94@gmail.com',
  '260324_2_step_bi_akg_to_hopa',
  '5f0f5291d8cc0b12ce4b74d4230e857bd0ca533d',
  1,
  '2026-03-27T11:04:43.503765'),
 ('stefanpate94@gmail.com',
  '260327_target_3hpa',
  'efbda24cd78bf8eb73f8a83a800d22e30ff5bf2e',
  0,
  '2026-03-28T16:59:22.694857'),
 ('stefanpate94@gmail.com',
  '260327_target_3hpa',
  'c9c16b750dad653da02911ca946bdc605fe2867b',
  0,
  '2026-03-28T16:59:22.694857'),
 ('Yash',
  '260327_target_3hpa',
  '1881af6948d5ae161c65c2c1ff4697961d68770f',
  0,
  '2026-04-07T14:30:59.643574'),
 ('stefanpate94@gmail.com',
  '260327_target_3hpa',
  'ac38f05c4ee4e5cd856007c283752e414f45f009',
  0,
  '2026-03-28T16:59:22.694857'),
 ('Yash',
  '260327_target_3hpa',
  

In [10]:
# Update the username in stefan_rows_by_table from "Stefan" to "stefanpate94@gmail.com"
new_username = "stefanpate94@gmail.com"

for table_name, rows in stefan_rows_by_table.items():
    for row in rows:
        # Convert tuple to list, update username (first element), convert back to tuple
        row_list = list(row)
        row_list[0] = new_username
        updated_row = tuple(row_list)
        
        # Update in the database
        conn.execute(
            f'UPDATE "{table_name}" SET username = ? WHERE username = ?',
            (new_username, "Stefan"),
        )

conn.commit()
print("Username updated successfully for all rows")


Username updated successfully for all rows
